# **ppkt2synergy Workflow**

This notebook demonstrates a complete workflow for analyzing phenopacket data using the `ppkt2synergy` package. It covers data loading, feature matrix construction, and downstream analysis including correlation and synergy detection between phenotypic features.

---

## Overview

The `ppkt2synergy` package is designed to transform phenopacket data into a structured representation suitable for statistical analysis.

At a high level, the workflow consists of:

1. Parsing phenopacket data
2. Constructing binary feature matrices for HPO terms and diseases
3. Performing pairwise **correlation analysis**
4. Quantifying higher-order relationships via **synergy analysis**
5. Visualizing the results

The goal is to uncover not only individual phenotype–phenotype associations, but also **interactions between phenotypic features** that may carry additional biological or diagnostic significance.

---

## Correlation Analysis

Correlation analysis measures the **pairwise association between two HPO terms** across patients.

Intuitively, it answers:

> *Do two phenotypic features tend to co-occur in the same patients?*

Depending on the implementation (using phi coefficient, Spearman rank correlation, and Kendall’s tau), correlation values typically fall into a standardized range:

* **Range**:

  * Phi coefficient, Spearman, and Kendall: **[-1, 1]**

* **Interpretation**:

  * **Positive correlation** → two HPO terms tend to appear together
  * **Negative correlation** → presence of one term is associated with absence of another
  * **Near 0** → little to no association

Correlation captures pairwise relationships between phenotypic features, but does not explicitly model condition-specific interactions involving additional variables such as disease context.

---

## Synergy Analysis

Synergy analysis goes beyond correlation by quantifying **higher-order interactions** between features.

In this context, synergy measures:

> *Whether the joint presence of two HPO terms provides more information about a disease than each term individually.*

* **Key idea**:

  * Two features may show weak or no correlation
  * But **together**, they can strongly indicate a specific disease

  * **Typical value range**:

  * Synergy is defined as a difference of mutual information terms and therefore typically ranges around **0**, with both positive and negative values possible.

  * In practice:

    * **> 0** → positive synergy (complementary information)
    * **≈ 0** → independent contributions
    * **< 0** → redundancy (overlapping information)

* **Interpretation**:

  * **Positive synergy** → the joint state of two HPO terms provides more information about the disease than the sum of their individual contributions

  * **Near-zero synergy** → the two terms contribute independently to the disease signal

  * **Negative synergy** → the two terms carry redundant information about the disease

Synergy is particularly useful for identifying:

* combinatorial phenotypes that jointly characterize disease states
* condition-specific interactions between phenotypic features
* non-linear and non-additive biological patterns

---


## Install Dependencies

To get started, install the required libraries:

```bash
pip install ppkt2synergy gpsea
```

In [ ]:
from ppkt2synergy import load_phenopackets,PhenopacketDatasetAssembler,HPOStatisticsAnalyzer,SynergyAnalyzer,CorrelationType
from gpsea.model import VariantEffect

### **Analysis Between HPO Terms and Target in a Single Cohort**

#### Overview

This section demonstrates the preparation and analysis of phenotypic data within a single cohort. It covers:

1. Loading phenopacket data for a specific gene (e.g., `"FBN1"`).  
2. Assembling patient-level datasets using `PhenopacketDatasetAssembler`.  
3. Extracting binary matrices for:
   - **HPO terms** (phenotypic features)
   - **Target variables** (e.g., disease status or variant effects)  
4. Preparing these matrices for downstream correlation and synergy analyses.

#### Feature Representation

Internally, the data is represented as **binary matrices**:

- **Rows**: individual patients  
- **Columns**: 
  - HPO terms (phenotypic features)  
  - Diseases or target variables (used as conditioning variables in downstream analysis)

Each entry encodes the status of a feature:

- `1` → observed  
- `0` → explicitly excluded  
- `NaN` → unknown or not reported

This representation provides a consistent, analysis-ready format suitable for:

- Pairwise correlation analysis between HPO terms  
- Condition-specific synergy analysis between HPO features and target variables
---

In [2]:
phenopackets = load_phenopackets("FBN1")
assembler = PhenopacketDatasetAssembler(phenopackets)
mane_tx_id = 'NM_000138.5'
variant_effect_type = VariantEffect.MISSENSE_VARIANT
hpo_data, target_data = assembler.build(mane_tx_id=mane_tx_id, variant_effect_type=variant_effect_type, threshold=0.9)

Individuals Processed: 100%|██████████| 143/143 [00:01<00:00, 81.00 individuals/s]


### **Correlation Analysis Between HPO Terms**

This section demonstrates computing pairwise correlations between HPO terms in the cohort using the `HPOStatisticsAnalyzer` class.


In [ ]:
correlation_type = CorrelationType.SPEARMAN  # Can be "kendall"or "phi"
analyzer = HPOStatisticsAnalyzer(
    hpo_data = hpo_data,
    min_individuals_for_correlation_test=30,
    min_cooccurrence_count=1)

analyzer.compute_correlation_matrix(
    correlation_type=correlation_type,
    n_jobs=32,
)

Calculating pairwise correlation: 100%|██████████| 606/606 [00:06<00:00, 98.78it/s] 


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,p_value_corrected,Count_00,Count_01,Count_10,Count_11,n_patients,n_pmids
0,HP:0000023,Inguinal hernia,HP:0000098,Tall stature,0.269191,1.362644e-01,3.969441e-01,9,18,0,5,32,1
1,HP:0000023,Inguinal hernia,HP:0000218,High palate,-0.130243,4.774010e-01,7.950254e-01,17,10,4,1,32,1
2,HP:0000023,Inguinal hernia,HP:0000268,Dolichocephaly,-0.077766,6.722574e-01,8.495624e-01,19,8,4,1,32,1
3,HP:0000023,Inguinal hernia,HP:0000272,Malar flattening,-0.111111,5.449055e-01,8.311903e-01,25,2,5,0,32,1
4,HP:0000023,Inguinal hernia,HP:0000490,Deeply set eye,0.051852,7.780677e-01,8.798076e-01,23,4,4,1,32,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,HP:0004322,Short stature,HP:0004942,Aortic aneurysm,-0.827131,1.078983e-19,2.168755e-17,7,34,33,0,74,5
398,HP:0004322,Short stature,HP:0004959,Descending thoracic aorta aneurysm,-0.237508,1.119779e-01,3.410236e-01,12,1,33,0,46,3
399,HP:0004322,Short stature,HP:0030053,Stiff skin,0.331133,1.111665e-02,6.206797e-02,50,7,0,1,58,4
400,HP:0004942,Aortic aneurysm,HP:0008848,Moderately short stature,-0.373288,1.458507e-03,1.221499e-02,27,9,34,0,70,5


After computing the correlation matrix, results can be filtered and saved for downstream analysis using the `save_correlation_results` method.
- **alpha** (float) : Significance level for **uncorrected p-values**.  
  - Only correlations with p-values ≤ `alpha` are considered statistically significant.  
  - This is useful for a first-pass filtering of potentially interesting feature pairs.

- **corrected_alpha** (float) : Significance level after **multiple testing correction** (e.g., Bonferroni or FDR).  
  - Adjusts for the large number of pairwise tests to control the false discovery rate.  
  - Only correlations with **corrected p-values ≤ `corrected_alpha`** are retained.  
  - Ensures that identified associations remain significant when accounting for multiple comparisons.

In [ ]:
analyzer.save_correlation_results(
    abs_threshold=0.55,
    alpha=1.0,
    corrected_alpha=1.0,
    output_file="correlation_results_FBN1.xlsx"
)

After computing and filtering pairwise HPO term correlations, a heatmap can be generated to visualize both correlation magnitude and statistical significance.


In [11]:
fig = analyzer.plot_correlation_heatmap_with_significance( 
    stats_name = correlation_type.value,
    abs_threshold=0.55,
    alpha=0.05,
    corrected_alpha=0.1,
    title_name="Cohort FBN1",
)
fig.show()
analyzer.save_correlation_heatmap(
    fig,
    output_file="correlation_heatmap_FBN1.html"
)   

### **Synergy Analysis Between HPO Terms and Target Variables**

This section demonstrates computation of **pairwise synergy** between HPO terms with respect to a target variable (e.g., disease status or variant effect) using the `SynergyAnalyzer` class.

In [3]:
target = target_data.variant_effect_df
synergy_analyzer=SynergyAnalyzer(
    hpo_data=hpo_data, 
    target=target,
    n_permutations=1000,
    min_individuals_for_synergy_calculation=40,
    random_state=40)
matrix= synergy_analyzer.compute_synergy_matrix(n_jobs=32)

Calculating pairwise synergy:   0%|          | 0/298 [00:00<?, ?it/s]

Calculating pairwise synergy: 100%|██████████| 298/298 [00:44<00:00,  6.72it/s]


In [4]:
synergy_analyzer.save_synergy_results(
    synergy_threshold=0.0,
    alpha=1.0,
    corrected_alpha=1.0,
    output_file='synergy_matrix_FBN1.csv',
)

In [ ]:
fig = synergy_analyzer.plot_synergy_heatmap(
    synergy_threshold= 0.08,
    alpha=0.05,
    corrected_alpha=0.1,
    target_name=target.columns[0],
)
fig.show()
synergy_analyzer.save_synergy_heatmap(
    fig,    
    output_file="synergy_heatmap_FBN1.html"
)

### **Analysis for Terms Across Multiple Cohorts**

This section demonstrates preparing HPO and target matrices for **multiple gene cohorts** simultaneously.

In [7]:
multi_cohort_names = ["TGFBR1","TGFBR2","SMAD3","TGFB2","TGFB3","SMAD2"]
phenopackets_multi = load_phenopackets(multi_cohort_names)
assembler_multi = PhenopacketDatasetAssembler(phenopackets_multi)
multi_mane_tx_ids = ['NM_004612.4','NM_003242.6','NM_005902.4','NM_003238.6','NM_003239.5','NM_005901.6']
multi_varsiant_effect_type = VariantEffect.MISSENSE_VARIANT
hpo_data_multi, target_data_multi = assembler_multi.build(mane_tx_id=multi_mane_tx_ids, variant_effect_type=multi_varsiant_effect_type, threshold=0.9)

Individuals Processed: 100%|██████████| 277/277 [00:05<00:00, 52.25 individuals/s]


#### Correlation Analysis

In [8]:
correlation_type = CorrelationType.SPEARMAN  # Can be "kendall"or "phi"
analyzer_multi = HPOStatisticsAnalyzer(
    hpo_data = hpo_data_multi,
    min_individuals_for_correlation_test=30,
    min_cooccurrence_count=1)

analyzer_multi.compute_correlation_matrix(
    correlation_type=correlation_type,
    n_jobs=32,
)

Calculating pairwise correlation: 100%|██████████| 1198/1198 [00:08<00:00, 137.71it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,p_value_corrected,Count_00,Count_01,Count_10,Count_11,n_patients,n_pmids
0,HP:0000023,Inguinal hernia,HP:0000098,Tall stature,-0.034417,0.818354,0.920646,6,17,7,17,47,5
1,HP:0000023,Inguinal hernia,HP:0000193,Bifid uvula,0.142733,0.307934,0.659347,25,6,15,7,53,8
2,HP:0000023,Inguinal hernia,HP:0000218,High palate,-0.092934,0.418359,0.728182,15,30,14,19,78,7
3,HP:0000023,Inguinal hernia,HP:0000268,Dolichocephaly,-0.267261,0.153357,0.495815,6,10,9,5,30,3
4,HP:0000023,Inguinal hernia,HP:0000272,Malar flattening,0.097518,0.500487,0.786731,15,18,6,11,50,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1102,HP:0012499,Descending aortic dissection,HP:0033981,Vertebral artery tortuosity,-0.044901,0.804043,0.918426,30,2,1,0,33,2
1103,HP:0012727,Thoracic aortic aneurysm,HP:0033311,Abdominal aortic dissection,-0.198030,0.254141,0.619458,14,1,20,0,35,2
1104,HP:0031869,Recurrent joint dislocation,HP:0500093,Food allergy,0.454344,0.007904,0.106702,27,1,3,2,33,1
1105,HP:0033311,Abdominal aortic dissection,HP:0033981,Vertebral artery tortuosity,0.695971,0.000007,0.000693,31,1,0,1,33,2


In [9]:
fig = analyzer_multi.plot_correlation_heatmap_with_significance( 
    correlation_type.value,
    abs_threshold=0.55,
    alpha=0.05,
    corrected_alpha=0.1,
    title_name="Loeys-Dietz syndrome",
)
fig

#### Synergy Analysis

In [8]:
target_data_multi.disease_matrix.columns

Index(['Congenital heart defects, multiple types, 8, with or without heterotaxy',
       'Loeys-Dietz syndrome 1', 'Loeys-Dietz syndrome 2',
       'Loeys-Dietz syndrome 3', 'Loeys-Dietz syndrome 4',
       'Loeys-Dietz syndrome 5', 'Loeys-Dietz syndrome 6',
       'Multiple self-healing squamous epithelioma, susceptibility to'],
      dtype='object')

### **Rationale for Filling NaN in Multi-Cohort Disease Targets**

- In a multi-cohort dataset, a given disease may be **observed only in a subset of cohorts**.  
- HPO/target matrices use `NaN` to indicate **unknown or unreported status**.  

- If `NaN` values are left as-is:
  - Entire rows for a disease target could be removed during analysis if all non-missing entries are filtered.  
  - This may result in the target having **only 1s and no valid comparison**, making **synergy computation impossible**.

- Using `fillna(0)` ensures:
  - Patients without reported information for this disease are treated as **not observed**.  
  - Preserves sufficient **variability in the target** for statistical analyses like synergy computation.

#### Notes

- This preprocessing step **avoids artificial loss of data** in multi-cohort analyses.  
- It allows **robust computation of target-conditioned synergy** even when a disease is only present in some cohorts.

In [ ]:
# Select disease target from multi-cohort target matrix
multi_target = target_data_multi.disease_matrix['Loeys-Dietz syndrome 2'].fillna(0)  

# Initialize synergy analyzer for multiple cohorts
multi_synergy_analyzer = SynergyAnalyzer(
    hpo_data=hpo_data_multi,
    target=multi_target,
    n_permutations=1000,
    min_individuals_for_synergy_calculation=40,
    random_state=40,
)

# Compute synergy values
multi_synergy_analyzer.compute_synergy_matrix(n_jobs=32)

Calculating pairwise synergy: 100%|██████████| 706/706 [02:04<00:00,  5.68it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,synergy,p_value,p_value_corrected,Count_00_y0,Count_01_y0,Count_10_y0,Count_11_y0,N_y0,Count_00_y1,Count_01_y1,Count_10_y1,Count_11_y1,N_y1,n_patients,n_pmids
0,HP:0000023,Inguinal hernia,HP:0000098,Tall stature,-0.016165,0.761,0.933677,6,10,7,16,39,0,7,0,1,8,47,5
1,HP:0000023,Inguinal hernia,HP:0000193,Bifid uvula,-0.008933,0.532,0.837763,17,3,14,6,40,8,3,1,1,13,53,8
2,HP:0000023,Inguinal hernia,HP:0000218,High palate,0.006429,0.239,0.682379,12,22,13,19,66,3,8,1,0,12,78,7
3,HP:0000023,Inguinal hernia,HP:0000272,Malar flattening,0.005346,0.282,0.717925,12,10,5,10,37,3,8,1,1,13,50,4
4,HP:0000023,Inguinal hernia,HP:0000316,Hypertelorism,0.001734,0.656,0.883148,23,11,19,14,67,0,0,1,0,1,68,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,HP:0010648,Dermal translucency,HP:0012385,Camptodactyly,-0.008700,0.624,0.870466,29,1,19,2,51,2,2,3,2,9,60,3
607,HP:0010648,Dermal translucency,HP:0012727,Thoracic aortic aneurysm,0.026104,0.118,0.644155,11,22,7,3,43,0,4,0,6,10,53,5
608,HP:0010648,Dermal translucency,HP:0033981,Vertebral artery tortuosity,0.000763,0.999,1.000000,28,2,20,0,50,4,0,2,0,6,56,3
609,HP:0010648,Dermal translucency,HP:0100775,Dural ectasia,-0.001349,0.440,0.804910,12,4,16,1,33,2,4,4,0,10,43,4


In [12]:
# Plot synergy heatmap
multi_synergy_analyzer.plot_synergy_heatmap(
    synergy_threshold= 0.08,
    alpha=0.05,
    corrected_alpha=0.1,
    target_name=multi_target.name
)

### **Comprehensive Analysis Across All Cohorts in Phenopacket Store**

This section demonstrates how to perform a **global analysis across all cohorts**, while focusing on a specific gene (e.g., `FBN1`) as the target condition.


#### Rationale

- The analysis defines a **binary target variable** indicating whether a patient is associated with the FBN1 cohort:

  - `1` → patient associated with FBN1  
  - `0` → patient from other cohorts  

- Specifically, synergy is evaluated as:

  > how pairs of HPO terms jointly contribute information about **FBN1 association**

- By restricting:
  - `hpo_data_all.hpo_matrix` to **HPO terms observed in the FBN1 cohort**, and aligning the corresponding **relationship mask**,  

  the analysis focuses on **FBN1-relevant phenotypic features**, while evaluating their interactions across the full dataset.

#### Ontology-Aware Filtering

- The `hpo_relationship_mask` is used to **exclude pairs of HPO terms that are directly related in the ontology** (e.g., parent–child relationships).

- This avoids detecting **spurious associations driven by hierarchical dependencies**, rather than true co-occurrence patterns.

- Using the mask derived from the **FBN1-specific feature set** ensures:
  - A consistent feature space  
  - Reduced redundancy due to ontology structure  
  - More reliable estimation of **condition-specific interactions (synergy)**

#### Notes

- This setup enables **condition-specific synergy analysis**, where:
  - The condition is **FBN1 association**  
  - The features are **FBN1-derived HPO terms**  
  - The population is **all available cohorts**

In [13]:
phenopackets_all = load_phenopackets()
assembler_all = PhenopacketDatasetAssembler(phenopackets_all)
hpo_data_all, target_data_all = assembler_all.build(threshold=1.0)

In [14]:
phenopackets_fbn1 = load_phenopackets(cohorts="FBN1")
assembler_fbn1 = PhenopacketDatasetAssembler(phenopackets_fbn1)
hpo_data_fbn1, target_data_fbn1 = assembler_fbn1.build(threshold=1.0)

In [ ]:
hpo_data_all.hpo_matrix = hpo_data_all.hpo_matrix[hpo_data_fbn1.hpo_matrix.columns]
hpo_data_all.hpo_relationship_mask = hpo_data_all.hpo_relationship_mask.loc[hpo_data_fbn1.hpo_matrix.columns, hpo_data_fbn1.hpo_matrix.columns]

#### Correlation Analysis

In [ ]:
correlation_type = CorrelationType.KENDALL  # Can be "kendall"or "spearman"
analyzer_all = HPOStatisticsAnalyzer(
    hpo_data = hpo_data_all,
    min_individuals_for_correlation_test=30,
    min_cooccurrence_count=1)

analyzer_all.compute_correlation_matrix(
    correlation_type=correlation_type,
    n_jobs=32,
)

Calculating pairwise correlation: 100%|██████████| 2430/2430 [00:10<00:00, 227.45it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,p_value_corrected,Count_00,Count_01,Count_10,Count_11,n_patients,n_pmids
0,HP:0000023,Inguinal hernia,HP:0000098,Tall stature,0.323184,6.051791e-06,6.042803e-05,114,44,13,26,197,15
1,HP:0000023,Inguinal hernia,HP:0000126,Hydronephrosis,0.133360,1.012654e-01,2.282415e-01,119,8,21,4,152,9
2,HP:0000023,Inguinal hernia,HP:0000218,High palate,0.131145,3.277019e-02,9.388844e-02,127,62,41,36,266,33
3,HP:0000023,Inguinal hernia,HP:0000219,Thin upper lip vermilion,0.199712,2.126791e-02,6.549219e-02,58,36,16,24,134,23
4,HP:0000023,Inguinal hernia,HP:0000268,Dolichocephaly,-0.013507,8.974748e-01,9.341384e-01,47,22,16,7,92,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2012,HP:0011968,Feeding difficulties,HP:0034391,Elbow contracture,0.193476,2.523656e-01,4.395695e-01,14,0,20,2,36,7
2013,HP:0012385,Camptodactyly,HP:0030799,Scaphocephaly,0.881917,3.638196e-08,6.671129e-07,35,0,1,4,40,7
2014,HP:0012385,Camptodactyly,HP:0034391,Elbow contracture,0.960691,2.089263e-37,2.107021e-34,147,0,2,29,178,33
2015,HP:0030799,Scaphocephaly,HP:6000264,Fetal tachycardia,1.000000,4.320463e-08,7.711835e-07,30,0,0,1,31,3


In [18]:
fig = analyzer_all.plot_correlation_heatmap_with_significance( 
    correlation_type.value,
    abs_threshold=0.8,
    alpha=0.05,
    corrected_alpha=0.1,
    title_name="FBN1 Cohort HPO Correlation (From Full Phenopacket Store)",
)
fig

#### Synergy Analysis

In [19]:
# Select disease target from multi-cohort target matrix
target_all = (target_data_all.cohort_info_df=="FBN1").astype(int)

# Initialize synergy analyzer for multiple cohorts
synergy_analyzer_all = SynergyAnalyzer(
    hpo_data=hpo_data_all,
    target=target_all,
    n_permutations=1000,
    min_individuals_for_synergy_calculation=40,
    random_state=40,
)

# Compute synergy values
synergy_analyzer_all.compute_synergy_matrix(n_jobs=32)

Calculating pairwise synergy: 100%|██████████| 2067/2067 [03:50<00:00,  8.95it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,synergy,p_value,p_value_corrected,Count_00_y0,Count_01_y0,Count_10_y0,Count_11_y0,N_y0,Count_00_y1,Count_01_y1,Count_10_y1,Count_11_y1,N_y1,n_patients,n_pmids
0,HP:0000023,Inguinal hernia,HP:0000098,Tall stature,0.015213,0.026,0.108657,105,26,13,21,165,9,18,0,5,32,197,15
1,HP:0000023,Inguinal hernia,HP:0000218,High palate,0.002020,0.205,0.367360,110,52,37,35,234,17,10,4,1,32,266,33
2,HP:0000023,Inguinal hernia,HP:0000268,Dolichocephaly,-0.006687,0.655,0.750870,28,14,12,6,60,19,8,4,1,32,92,10
3,HP:0000023,Inguinal hernia,HP:0000272,Malar flattening,-0.001867,0.401,0.551744,39,26,22,13,100,25,2,5,0,32,132,10
4,HP:0000023,Inguinal hernia,HP:0000490,Deeply set eye,-0.006117,0.596,0.712401,38,17,11,3,69,23,4,4,1,32,101,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1115,HP:0011003,High myopia,HP:0011220,Prominent forehead,-0.000034,0.252,0.413061,123,42,3,1,169,0,1,0,0,1,170,13
1116,HP:0011003,High myopia,HP:0011968,Feeding difficulties,-0.007692,0.014,0.070631,84,92,1,5,182,0,0,0,2,2,184,22
1117,HP:0011003,High myopia,HP:0012385,Camptodactyly,0.001231,0.076,0.204615,65,10,1,1,77,0,2,0,0,2,79,10
1118,HP:0012385,Camptodactyly,HP:0034391,Elbow contracture,-0.118104,0.000,0.000000,147,0,2,21,170,0,0,0,8,8,178,33


In [20]:
# Plot synergy heatmap
synergy_analyzer_all.plot_synergy_heatmap(
    synergy_threshold=0.08,
    alpha=0.05,
    corrected_alpha=0.1,
    target_name='is_FBN1'
)